In [1]:
from datasets import load_dataset
from collections import Counter

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("iberu/RunBugRun")

print(ds.keys())
print(ds['train'].column_names)
print(ds['train'][0])

C:\Users\Home\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dict_keys(['train', 'validation', 'test'])
['id', 'buggy_submission_id', 'fixed_submission_id', 'problem_id', 'user_id', 'buggy_code', 'fixed_code', 'labels', 'change_count', 'line_hunks', 'errors']
{'id': 6581, 'buggy_submission_id': 3585, 'fixed_submission_id': 3586, 'problem_id': 'p00000', 'user_id': 'u243053043', 'buggy_code': 'for i in range(1, 10):\n    for j in range(1, 10):\n        print("{} x {} = {}".format(i, j, i*j))', 'fixed_code': 'for i in range(1, 10):\n    for j in range(1, 10):\n        print("{}x{}={}".format(i, j, i*j))', 'labels': ['literal.string.change', 'call.arguments.change', 'io.output.change'], 'change_count': 1, 'line_hunks': 1, 'errors': None}


In [4]:
#investigating: distribution of labels for top 20 and the number of unique labels
label_counter = Counter()

for row in ds['train']: #iterates through all of the examples in the training only, counting each label
    if row['labels']:
        label_counter[row['labels'][0]] += 1

print('\nTop 20 Labels:')
for label, count in label_counter.most_common(20):
    print(label, count)

print('\n Number of Unique Primary Labels:', len(label_counter))


Top 20 Labels:
literal.number.integer.change 11636
call.add 10883
identifier.change 10300
assignment.value.change 10035
literal.string.change 9816
call.remove 9452
call.arguments.change 7901
expression.operator.compare.change 7812
control_flow.branch.if.condition.change 7122
expression.operation.binary.remove 5252
control_flow.loop.range.bounds.upper.change 4200
misc.opposites 4132
call.arguments.add 3560
expression.operator.arithmetic.change 3341
expression.operation.binary.add 2760
assignment.variable.change 2570
assignment.add 2083
assignment.change 1748
identifier.replace.add 1313
identifier.replace.remove 817

 Number of Unique Primary Labels: 59


In [ ]:
#investigating: how many labels per example
label_count_dist = Counter()
missing_labels = 0

for row in ds['train']:
    if row['labels'] is None:
        missing_labels += 1
        continue
    label_count_dist[len(row["labels"])] += 1

print("Labels per example:")
for num_labels, count in sorted(label_count_dist.items()):
    print(f'{num_labels} labels: {count}')

In [ ]:
#investigating: how many top level categories exist
unique_label_category = Counter()

for row in ds['train']:
    if row['labels'] == None:
            continue
    for label in row['labels']:
        category = label.split('.')[0]
        unique_label_category[category] += 1

for name, count in sorted(unique_label_category.items()):
    print(f'{name} - {count}')

In [ ]:
#investigating: checking if the dataset is all python by looking at random examples
for example in [1, 20, 50, 100, 200, 4000, 20000]:
     code = ds['train'][example]['buggy_code']
     print(f'{example} - Code: {code}')

In [ ]:
#investigating: the try-catch label. python has try and except
for row in ds['train']:
    if row['labels'] is None:
        continue
    for label in row['labels']:
        if label.split('.')[0] == 'try_catch':
            print(f"{label} - Code: {row['buggy_code']}")

In [2]:
#investigating: how many unique labels per example
unique_amount_label_counter = Counter()

for row in ds['train']:
    if row['labels'] is None:
        continue
    else:
        unique_label_in_row = set()
        for label in row['labels']:
            label = label.split('.')[0]
            unique_label_in_row.add(label)
            
        unique_amount_label_counter[len(unique_label_in_row)] += 1

for key, value in unique_amount_label_counter.most_common(5):
    print(f'Unique Identifier Amount: {key} - Frequency: {value}')

Unique Identifier Amount: 1 - Frequency: 47378
Unique Identifier Amount: 2 - Frequency: 29432
Unique Identifier Amount: 3 - Frequency: 28923
Unique Identifier Amount: 4 - Frequency: 11945
Unique Identifier Amount: 0 - Frequency: 6391


In [3]:
#investigating: rows with no labels
rows_with_empty_label = []
for row in ds['train']:
    if row['labels'] == None:
        rows_with_empty_label.append((row['buggy_code'], row['fixed_code']))

print(f'This is the number of rows in the rows without any labels: {len(rows_with_empty_label)}')

for i in range(5): #the range can be changed as long as it is less than the len of the list
    print(f'Buggy Code {i}:\n {rows_with_empty_label[i][0]}\n Fixed Code {i}:\n {rows_with_empty_label[i][1]}')

This is the number of rows in the rows without any labels: 3753
Buggy Code 0:
 for i in range(1,10):
		for i in range(1,10):
			print("%dx%d=%d"%(i,j,i*j))
 Fixed Code 0:
 for i in range(1,10):
		for j in range(1,10):
			print("%dx%d=%d"%(i,j,i*j))
Buggy Code 1:
 for i in range(10):
    for s in range(10):
        print(str(i)+"x"+str(s)+"="+str(i*s))
 Fixed Code 1:
 for i in range(1,10):
    for s in range(1,10):
        print(str(i)+"x"+str(s)+"="+str(i*s))
Buggy Code 2:
 j = []
for i in range(10) :
    j.append(int(input()))
j.reverse()
for k in range(3) :
    print(j[k])
 Fixed Code 2:
 j = []
for i in range(10) :
    j.append(int(input()))
j.sort(reverse = True)
for k in range(3) :
    print(j[k])
Buggy Code 3:
 def main():
    l = []
    for _ in range(10):
        l.append(input())
    l = sorted(l, reverse=True)
    for i in range(3):
        print(l[i])
    return None

if __name__ == '__main__':
    main()
 Fixed Code 3:
 def main():
    l = []
    for _ in range(10):
       